# M49 --- external validation of the best ICBHI model on **HF_Lung_V1**

The best checkpoint on the corrected ICBHI official split is run, **unchanged**, over a
corpus it has never seen, and scored with the same official ICBHI metric. No fine-tuning,
no threshold tuning, no target label used to fit anything in the headline number.

**HF_Lung_V1** --- 9,765 fifteen-second recordings from Taiwan, natively at
4 kHz, labelled with breath phases and adventitious-sound spans. The shift against ICBHI is
population, hardware **and** bandwidth: 4 kHz sampling puts Nyquist at 2,000 Hz, exactly the
model's mel `fmax`, so the top of the band arrives attenuated by the source anti-aliasing.
That is a real confound and it is recorded in `dataset_info.native_sample_rates_hz`.

**Unit of analysis.** HF_Lung_V1 annotates breath *phases*, not cycles, so each inhalation is
paired with the exhalation that follows it within 1 s to form one ICBHI-style cycle. An
unpaired phase becomes a cycle on its own rather than being dropped --- discarding lone phases
would throw away the breaths at the clip boundaries, which is where a 15 s window cuts a cycle
in half, and that loss is not random with respect to the label.

**Taxonomy.** A cycle is Crackle if a `D` span overlaps it by >= 50 ms, Wheeze if a
`Wheeze` / `Stridor` / `Rhonchi` span does, Both if both, Normal otherwise --- the same
"contains the sound" rule ICBHI uses. `I` and `E` build the cycle and are not labels. An
unmapped token raises.

**Confidence interval:** **recording-level**, not patient-level. HF_Lung_V1 filenames carry a
timestamp and no patient ID, so the bootstrap resamples recording sessions (all
`trunc_...-LX_N` slices of one session stay together). Calling it patient-level would
overstate what the resampling controls for, and the results JSON names the unit explicitly.

## Before you press Run All

| | |
|---|---|
| **Accelerator** | GPU T4 (CPU works but the pass is slower) |
| **Internet** | **ON** --- the notebook downloads HF_Lung_V1 from GitLab |
| **Add Data** | `vbookshelf/respiratory-sound-database` (ICBHI --- needed for the verification gate) |
| **Add Data** | the checkpoint: upload `Asif's/M45/best_M45_P3.pth` as a private Kaggle dataset |
| **Runtime** | roughly 10 min download + 10 min evaluation on a T4 |

### The gate

Before a single external number is computed, the notebook re-scores the checkpoint on the
2,636 ICBHI test cycles and **asserts it reproduces the score stored inside the checkpoint**
(P3: 0.5764). That proves this notebook's preprocessing, channel expansion and ImageNet
normalisation are the training run's --- so a low external score can be read as domain shift
rather than as a bug in the harness. A mismatch aborts the notebook.

### Bring back

`results_M49_*.json`, `preds_M49_*.npy`, `confusion_M49_*.png` --- zipped in the last cell.
Commit them into `M49_cross_dataset/`.


## 1. Environment

In [ ]:
!pip -q install librosa==0.10.2 soundfile
import glob, json, os, subprocess, sys, time
import numpy as np
print(sys.version)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

WORK = "/kaggle/working/M49"
os.makedirs(WORK, exist_ok=True)

# Set to a row count for a 2-minute smoke run; None for the real thing.
SMOKE = None
# The frozen-feature probe uses target labels, so it is NOT a zero-shot number. It is
# reported in its own block and answers a question the headline number cannot: whether the
# representation is useless here, or only the ICBHI-fitted decision boundary is.
RUN_PROBE = True


## 2. Locate the checkpoint and ICBHI

By glob, not by a hard-coded path --- Kaggle's dataset mirrors differ in layout, and a wrong path that silently resolves to an empty directory is a failure mode this project has already been bitten by.

In [ ]:
def find_one(pattern, what, hint=""):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"could not find {what} with {pattern!r}. {hint}")
    return hits[0]

# Change this to evaluate a different checkpoint. Any M45 / M22_v2 checkpoint works: the
# loader reads the preprocessing flags out of the checkpoint's own cfg, and the ICBHI gate
# then holds it to the score IT reports, not to a hard-coded one.
CKPT_NAME = "best_M45_P3.pth"
CKPT = find_one(f"/kaggle/input/**/{CKPT_NAME}", "the checkpoint",
                f"Upload Asif's/M45/{CKPT_NAME} as a private Kaggle dataset and attach it, "
                "or set CKPT_NAME to whichever checkpoint you attached.")
ICBHI_AUDIO = os.path.dirname(find_one(
    "/kaggle/input/**/audio_and_txt_files/*.wav", "the ICBHI audio",
    "Add Data -> vbookshelf/respiratory-sound-database."))
ICBHI_SPLIT = find_one("/kaggle/input/**/ICBHI_challenge_train_test.txt",
                       "the official ICBHI split file",
                       "It ships with the same dataset. A notebook that cannot find it "
                       "must raise, never fall back (Model_Training_Protocol.md 1.1).")
print("checkpoint  :", CKPT)
print("ICBHI audio :", ICBHI_AUDIO, len(glob.glob(ICBHI_AUDIO + "/*.wav")), "wavs")
print("ICBHI split :", ICBHI_SPLIT)


## 2b. Get HF_Lung_V1

The corpus ships as split 7-Zip archives on GitLab (`train.7z.001`--`010`,
`test.7z.001`--`003`, about 1.1 GB total). Tried in order: an attached Kaggle dataset, then a
direct download. Needs **Internet ON**.

Set `HFL_PARTS = ("test",)` for a faster first pass over the test half only --- but then say
so, because it is a different corpus subset from the full run.

In [ ]:
HFL_PARTS = ("train", "test")     # ("test",) for a quicker first pass
HFL_ROOT = None

hits = sorted(glob.glob("/kaggle/input/**/*_label.txt", recursive=True))
if hits:
    # commonpath over every label file, so a mirror that flattened train/ and test/ into one
    # directory resolves just as well as one that kept them
    HFL_ROOT = os.path.commonpath([os.path.dirname(p) for p in hits])
    print("using the attached dataset:", HFL_ROOT, len(hits), "label files")
else:
    if subprocess.call(["which", "7z"]) != 0:
        subprocess.call(["apt-get", "-qq", "install", "-y", "p7zip-full"])
    assert subprocess.call(["which", "7z"]) == 0, (
        "7z is not available and could not be installed. Extract HF_Lung_V1 locally, "
        "upload it as a Kaggle dataset, and attach it - the cell above will find it.")
    HFL_ROOT = "/kaggle/working/HF_Lung_V1"
    os.makedirs(HFL_ROOT, exist_ok=True)
    base = "https://gitlab.com/techsupportHF/HF_Lung_V1/-/raw/master"
    n_parts = {"train": 10, "test": 3}
    for part in HFL_PARTS:
        if os.path.isdir(os.path.join(HFL_ROOT, part)):
            print(f"{part}/ already extracted, skipping")
            continue
        t0 = time.time()
        for i in range(1, n_parts[part] + 1):
            name = f"{part}.7z.{i:03d}"
            dst = os.path.join(HFL_ROOT, name)
            if not os.path.exists(dst):
                subprocess.check_call(["wget", "-q", "-O", dst,
                                       f"{base}/{name}?inline=false"])
            print(f"  {name} {os.path.getsize(dst)/1e6:.0f} MB")
        subprocess.check_call(["7z", "x", "-y", f"-o{HFL_ROOT}",
                               os.path.join(HFL_ROOT, f"{part}.7z.001")],
                              stdout=subprocess.DEVNULL)
        for f in glob.glob(os.path.join(HFL_ROOT, f"{part}.7z.*")):
            os.remove(f)
        print(f"  {part} ready in {(time.time()-t0)/60:.1f} min")

wavs = glob.glob(os.path.join(HFL_ROOT, "**", "*.wav"), recursive=True)
labs = glob.glob(os.path.join(HFL_ROOT, "**", "*_label.txt"), recursive=True)
assert wavs and labs, (f"nothing under {HFL_ROOT}. Turn Internet ON, or attach an "
                       "HF_Lung_V1 mirror as a dataset.")
print("HF_Lung_V1 root:", HFL_ROOT, "|", len(wavs), "wav |", len(labs), "label files")
print("a label file, verbatim:")
print(open(sorted(labs)[0]).read()[:300])


## 3. The two modules

`m45_ablation.py` is embedded **verbatim** --- it is the module that trained this checkpoint,
and it supplies the mel parameters, the official metric and the corrected-split loader.
`m49_xval.py` imports it rather than re-implementing any of it, because a second
implementation of the mel parameters would make every cross-dataset delta a measurement of
the gap between two scripts.

In [ ]:
M45_ABLATION_SRC = r'''"""
M45 — Component + preprocessing ablation on the best model (RTK requirements 10 AND 1)

BASELINE: M22_v2 — MobileNetV2 + SpecAugment, official ICBHI 0.5602 [0.5081, 0.6137] on
`official_60_40_patient_independent_corrected`. One variable changes per row; everything
else is held at the M22_v2 recipe.

WHY ONE SCRIPT FOR TWO REQUIREMENTS
    RTK_requirements.md section 10 says so directly: "Merge M45 and M46 into one ablation
    section in the report; they are the same kind of evidence."

THE THREE MISSING STAGES ARE NOW ROWS, BUT THEY ADD RATHER THAN REMOVE
    RTK section 3b lists six preprocessing rows. Three of them ablate stages that did not
    exist in this pipeline: an explicit band-pass filter, denoising, and per-cycle
    amplitude normalisation. Section 3a asks for those stages to be written first.

    They are now implemented (`bandpass`, `denoise`, `ampnorm` in the config), and they are
    OFF in BASE. That is deliberate. Turning any of them on in BASE would change A0, and
    every delta in this table is measured against A0 — a silent baseline change would
    invalidate the seven rows already run and the paper table built from them.

    So P1-P3 are ADD rows, not REMOVE rows: each turns one stage on and reports what it
    buys. Read their delta in the opposite direction to A1-A6 and P4-P5. A positive delta
    on an ADD row is a recommendation to adopt the stage; a negative one is evidence that
    the stage is not missing from this pipeline by oversight but by merit.

A ROW THE SPEC DID NOT ASK FOR, ADDED ON EVIDENCE
    P4_zeropad. M44 measured attribution consistency across the wrap-padded repetitions of
    each cycle and found it low (mean r=0.098; only 32% of cycles above r=0.5). Every ICBHI
    cycle is shorter than the 8 s input (median 2.42 s), so the pipeline tiles it ~3x with
    `np.tile` — a hard concatenation that leaves a discontinuity at every seam.

    Cyclic padding to 8 s is standard practice on ICBHI, so this is not an idiosyncratic
    choice; but padding-driven shortcut learning is a documented failure mode in audio
    classification, and some published implementations tile WITH a fade in/out precisely to
    avoid seam artefacts. This row tests whether the padding scheme is doing work the
    acoustics should be doing. It is the one row motivated by a measurement rather than by
    a checklist.

ROWS
    A0  baseline            = M22_v2 (already run, reused)
    A1  - SpecAugment       = M3_v2  (already run, reused)
    A2  - ImageNet init     random initialisation
    A3  - class weighting   plain unweighted CE
    A4  frozen backbone     train the classifier only
    A5  64 mels             half the frequency resolution
    A6  4 s cycles          half the temporal context
    P4  zero-padding        pad with silence instead of tiling the cycle
    P5  - min-max norm      drop the per-spectrogram [0,1] rescale

USAGE
    python m45_ablation.py --row A2                 # one row
    python m45_ablation.py --all                    # every unrun row, sequentially
    python m45_ablation.py --summarise              # build the table from what exists
"""
from __future__ import annotations

import argparse
import datetime
import glob
import json
import math
import os
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
REPO = os.path.abspath(os.path.join(HERE, "..", ".."))
CLASSES = ["Normal", "Crackle", "Wheeze", "Both"]
LABEL_OF = {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}
SR = 16000

BASE = dict(n_mels=128, duration_s=8.0, padding="wrap", minmax=True,
            bandpass=False, denoise=False, ampnorm=False,
            pretrained=True, freeze=False, class_weighted=True, specaug=True,
            lr=5e-4, batch_size=16, epochs=40, dropout=0.3, seed=42)

ROWS = {
    "A2": dict(pretrained=False, _desc="- ImageNet pre-training (random init)"),
    "A3": dict(class_weighted=False, _desc="- class-weighted loss (plain CE)"),
    "A4": dict(freeze=True, _desc="frozen backbone, classifier only"),
    "A5": dict(n_mels=64, _desc="64 mels instead of 128"),
    "A6": dict(duration_s=4.0, _desc="4 s cycles instead of 8 s"),
    "P4": dict(padding="zero", _desc="zero-padding instead of cyclic tiling"),
    "P5": dict(minmax=False, _desc="- per-spectrogram min-max normalisation"),
    # ADD rows - see the docstring. Delta reads in the opposite direction to the rows above.
    "P1": dict(bandpass=True, _desc="+ band-pass filter (50-2000 Hz Butterworth)"),
    "P2": dict(denoise=True, _desc="+ spectral-gating denoising"),
    "P3": dict(ampnorm=True, _desc="+ per-cycle peak amplitude normalisation"),
}
ADD_ROWS = {"P1", "P2", "P3"}
REUSED = {
    "A0": (os.path.join(REPO, "Asif's", "M22_v2", "Results", "results_M22_v2.json"),
           "baseline (M22_v2): MobileNetV2 + SpecAugment"),
    "A1": (os.path.join(REPO, "Asif's", "M3_v2", "Results", "results_M3_v2.json"),
           "- SpecAugment (M3_v2)"),
}
NOT_APPLICABLE = {}   # P1-P3 were the entries here; they are ADD rows now (see docstring)


# ---------------------------------------------------------------- data
def corrected_split_index(audio_dir, split_file):
    split = {}
    for line in open(split_file):
        t = line.replace("\t", " ").replace(",", " ").split()
        if len(t) >= 2 and t[1].lower() in ("train", "test"):
            split[t[0].replace(".wav", "")] = t[1].lower()
    assert len(split) == 920, f"{len(split)} != 920 recordings"
    pid = lambda s: int(s.split("_")[0])
    sides = {}
    for s, v in split.items():
        sides.setdefault(pid(s), set()).add(v)
    overlap = {p for p, v in sides.items() if len(v) > 1}
    split = {s: ("train" if pid(s) in overlap else v) for s, v in split.items()}
    assert sum(1 for v in split.values() if v == "train") == 551, "corrected split wrong"

    rows = []
    for wav in sorted(glob.glob(os.path.join(audio_dir, "*.wav"))):
        stem = os.path.splitext(os.path.basename(wav))[0]
        txt = os.path.join(audio_dir, stem + ".txt")
        if stem not in split or not os.path.exists(txt):
            continue
        for line in open(txt):
            p = line.split()
            if len(p) >= 4:
                rows.append({"wav": wav, "stem": stem, "patient_id": pid(stem),
                             "start": float(p[0]), "end": float(p[1]),
                             "label": LABEL_OF[(int(p[2]), int(p[3]))],
                             "split": split[stem]})
    return rows


def butter_bandpass(a, lo=50.0, hi=2000.0, order=4):
    """Zero-phase 4th-order Butterworth band-pass (RTK section 3a stage 2).

    filtfilt, not lfilter: a causal filter shifts transients in time, and crackles are
    transients whose timing is the thing being measured.
    """
    from scipy.signal import butter, filtfilt
    nyq = SR / 2.0
    hi = min(hi, nyq * 0.999)
    b, a_ = butter(order, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a_, a).astype(np.float32)


def peak_normalise(a):
    """Per-cycle peak normalisation (RTK section 3a stage 5) - removes device gain."""
    m = float(np.max(np.abs(a)))
    return (a / m).astype(np.float32) if m > 1e-8 else a


def spectral_gate(power, floor_pct=25.0, over=2.0):
    """Spectral-subtraction denoising on the power spectrogram (RTK section 3a stage 6).

    The noise floor is estimated per frequency bin as a low percentile over time, which
    assumes the noise is stationary within a cycle and the adventitious sound is not.
    Subtracted with an over-subtraction factor and floored at zero rather than at a small
    epsilon, so a bin that is pure noise becomes silent instead of becoming a small
    positive number the log stretches back into visible structure.

    The percentile and the factor are tied together and were set by the self-test, not by
    taste. Spectrogram power in a stationary bin is roughly exponential, so the 25th
    percentile sits near 0.29 of the mean; over=2.0 therefore subtracts a little over half
    the expected floor. An earlier setting (10th percentile, over=1.5) removed under 5% of
    it and would have shipped as a stage that measurably did nothing. Pushing harder is
    possible but buys musical noise, which a CNN can learn as texture, so this stays
    deliberately conservative.
    """
    noise = np.percentile(power, floor_pct, axis=1, keepdims=True)
    return np.maximum(power - over * noise, 0.0)


def log_mel(wav, start, end, cfg):
    """The M22 preprocessing, with every ablatable stage switchable."""
    import librosa
    n_mels, dur = cfg["n_mels"], cfg["duration_s"]
    n_samples = int(SR * dur)
    n_frames = 1 + n_samples // 160
    try:
        a, _ = librosa.load(wav, sr=SR, offset=start, duration=max(end - start, 0.05),
                            mono=True)
    except Exception as e:
        # A silent all-zero spectrogram here would be trained on and
        # scored as a real cycle. Fail instead of substituting
        # (Model_Training_Protocol.md section 1.2).
        raise RuntimeError("failed to load audio") from e
    if len(a) == 0:
        # Empty decode is a failed read, not a silent zero cycle.
        raise RuntimeError(f"empty audio decoded from {wav}")

    if cfg.get("bandpass"):
        a = butter_bandpass(a)

    if len(a) < n_samples:
        if cfg["padding"] == "wrap":                       # tile the cycle (the default)
            a = np.tile(a, math.ceil(n_samples / len(a)))[:n_samples]
        else:                                              # pad with silence
            a = np.pad(a, (0, n_samples - len(a)))
    else:
        a = a[:n_samples]

    if cfg.get("ampnorm"):
        a = peak_normalise(a)

    if cfg.get("denoise"):
        # Gate in the linear STFT domain, then project through the mel filterbank, so the
        # subtraction happens at the resolution the noise was estimated at.
        spec = np.abs(librosa.stft(a, n_fft=1024, hop_length=160, win_length=400)) ** 2
        m = librosa.feature.melspectrogram(S=spectral_gate(spec), sr=SR, n_mels=n_mels,
                                           n_fft=1024, fmin=50, fmax=2000)
    else:
        m = librosa.feature.melspectrogram(y=a, sr=SR, n_mels=n_mels, n_fft=1024,
                                           hop_length=160, win_length=400, fmin=50,
                                           fmax=2000, power=2.0)
    lm = librosa.power_to_db(m, ref=np.max)
    if cfg["minmax"]:
        lm = (lm - lm.min()) / (lm.max() - lm.min() + 1e-8)
    else:
        lm = lm / 80.0 + 1.0                               # keep dB roughly in [0,1]
    T = lm.shape[1]
    lm = np.pad(lm, ((0, 0), (0, n_frames - T))) if T < n_frames else lm[:, :n_frames]
    return lm[None].astype(np.float32)


def build_cache(rows, cfg, tag):
    # Every stage that changes the pixels must be in the key. A cache hit on a stale key
    # would train the new row on the old row's spectrograms and report it as a result.
    key = (f"{cfg['n_mels']}m_{cfg['duration_s']}s_{cfg['padding']}_{int(cfg['minmax'])}"
           f"_bp{int(cfg.get('bandpass', False))}_dn{int(cfg.get('denoise', False))}"
           f"_an{int(cfg.get('ampnorm', False))}")
    n_frames = 1 + int(SR * cfg["duration_s"]) // 160
    path = os.path.join(HERE, "cache", f"{tag}_{key}.npy")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    shape = (len(rows), 1, cfg["n_mels"], n_frames)
    if os.path.exists(path):
        return np.memmap(path, dtype=np.float16, mode="r", shape=shape)
    print(f"    building cache {os.path.basename(path)} ...")
    mm = np.memmap(path, dtype=np.float16, mode="w+", shape=shape)
    for i, r in enumerate(rows):
        mm[i] = log_mel(r["wav"], r["start"], r["end"], cfg).astype(np.float16)
        if (i + 1) % 1500 == 0:
            print(f"      {i+1}/{len(rows)}")
    mm.flush()
    return np.memmap(path, dtype=np.float16, mode="r", shape=shape)


# ---------------------------------------------------------------- metrics
def official(cm):
    cm = np.asarray(cm, float)
    sp = cm[0, 0] / cm[0].sum() if cm[0].sum() else float("nan")
    abn = cm[1:].sum()
    se = (cm[1, 1] + cm[2, 2] + cm[3, 3]) / abn if abn else float("nan")
    return float((se + sp) / 2), float(se), float(sp)


def patient_ci(y, pred, pid, n_boot=2000, seed=42):
    from sklearn.metrics import confusion_matrix
    uq = np.unique(pid); ix = {p: np.flatnonzero(pid == p) for p in uq}
    rng = np.random.default_rng(seed); v = []
    for _ in range(n_boot):
        s = np.concatenate([ix[p] for p in rng.choice(uq, len(uq), replace=True)])
        x = official(confusion_matrix(y[s], pred[s], labels=[0, 1, 2, 3]))[0]
        if x == x:
            v.append(x)
    return [round(float(z), 4) for z in np.percentile(v, [2.5, 97.5])]


# ---------------------------------------------------------------- run one row
def run_row(row_id, rows, args):
    import torch, torch.nn as nn, torchvision
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

    cfg = dict(BASE); cfg.update({k: v for k, v in ROWS[row_id].items() if not k.startswith("_")})
    desc = ROWS[row_id]["_desc"]
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    n_frames = 1 + int(SR * cfg["duration_s"]) // 160
    print(f"\n{'='*74}\n  {row_id}: {desc}\n{'='*74}")

    tr = [r for r in rows if r["split"] == "train"]
    te = [r for r in rows if r["split"] == "test"]
    Xtr, Xte = build_cache(tr, cfg, "train"), build_cache(te, cfg, "test")
    ytr = np.array([r["label"] for r in tr]); yte = np.array([r["label"] for r in te])
    pid = np.array([r["patient_id"] for r in te])

    def spec_aug(x):
        for _ in range(2):
            f = int(torch.randint(0, 25, (1,)).item())
            if 0 < f < x.shape[1]:
                f0 = int(torch.randint(0, x.shape[1] - f + 1, (1,)).item()); x[:, f0:f0+f, :] = 0
        for _ in range(2):
            t = int(torch.randint(0, 81, (1,)).item())
            if 0 < t < x.shape[2]:
                t0 = int(torch.randint(0, x.shape[2] - t + 1, (1,)).item()); x[:, :, t0:t0+t] = 0
        return x

    class DS(Dataset):
        def __init__(s, X, y, aug): s.X, s.y, s.aug = X, y, aug
        def __len__(s): return len(s.y)
        def __getitem__(s, i):
            x = torch.from_numpy(np.asarray(s.X[i], np.float32))
            return (spec_aug(x.clone()) if s.aug else x), torch.tensor(int(s.y[i]))

    dl_tr = DataLoader(DS(Xtr, ytr, cfg["specaug"]), batch_size=cfg["batch_size"],
                       shuffle=True, num_workers=0)
    dl_te = DataLoader(DS(Xte, yte, False), batch_size=cfg["batch_size"], num_workers=0)

    mean = torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)
    std = torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            w = "IMAGENET1K_V1" if cfg["pretrained"] else None
            s.features = torchvision.models.mobilenet_v2(weights=w).features
            if cfg["freeze"]:
                for p in s.features.parameters():
                    p.requires_grad = False
            s.gap = nn.AdaptiveAvgPool2d((1, 1)); s.dropout = nn.Dropout(cfg["dropout"])
            s.classifier = nn.Linear(1280, 4); s.norm = cfg["pretrained"]
        def forward(s, x):
            x = x.repeat(1, 3, 1, 1)
            if s.norm:
                x = (x - mean.to(x.device)) / std.to(x.device)
            return s.classifier(s.dropout(s.gap(s.features(x)).flatten(1)))

    torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    model = Net().to(dev)
    if cfg["class_weighted"]:
        c = np.maximum(np.bincount(ytr, minlength=4).astype(float), 1)
        w = c.sum() / (4 * c); w = w / w.mean()
        crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=dev))
    else:
        crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                           lr=cfg["lr"], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

    best, best_pred, best_ep, t0 = -1.0, None, 0, time.time()
    for ep in range(1, cfg["epochs"] + 1):
        model.train()
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev)
            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                loss = crit(model(x), y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        sch.step()
        model.eval(); pr = []
        with torch.no_grad():
            for x, _ in dl_te:
                pr.append(model(x.to(dev)).argmax(1).cpu().numpy())
        pr = np.concatenate(pr)
        sc = official(confusion_matrix(yte, pr, labels=[0, 1, 2, 3]))[0]
        if sc > best:
            best, best_pred, best_ep = sc, pr, ep
        if ep % 10 == 0 or ep == cfg["epochs"]:
            print(f"    ep {ep:02d}/{cfg['epochs']}  official {sc:.4f}  (best {best:.4f})")

    cm = confusion_matrix(yte, best_pred, labels=[0, 1, 2, 3])
    sc, se, sp = official(cm)
    doc = {"meta": {"model_id": f"M45_{row_id}", "row": row_id, "variable_changed": desc,
                    "baseline_model_id": "M22_v2",
                    "date_completed": datetime.datetime.now().strftime("%Y-%m-%d"),
                    "is_augmented": bool(cfg["specaug"]),
                    "notes": "M45 ablation row. One variable changed from the M22_v2 recipe."},
           "config": {k: v for k, v in cfg.items()},
           "dataset_info": {"dataset": "ICBHI_2017",
                            "split_method": "official_60_40_patient_independent_corrected",
                            "train_samples": len(tr), "test_samples": len(te),
                            "test_patients": int(len(np.unique(pid)))},
           "best_epoch": {"epoch": best_ep, "primary_metric": "icbhi_score_official",
                          "primary_metric_value": round(sc, 4)},
           "best_metrics": {"icbhi_score_official": round(sc, 4),
                            "icbhi_score_official_ci95": patient_ci(yte, best_pred, pid),
                            "icbhi_se_official": round(se, 4),
                            "icbhi_sp_official": round(sp, 4),
                            "accuracy": round(float(accuracy_score(yte, best_pred)), 4),
                            "f1_macro": round(float(f1_score(yte, best_pred, average="macro",
                                                             zero_division=0)), 4),
                            "confusion_matrix_raw": cm.tolist()},
           "efficiency": {"total_params": int(sum(p.numel() for p in model.parameters())),
                          "trainable_params": int(sum(p.numel() for p in model.parameters()
                                                      if p.requires_grad)),
                          "training_time_total_s": round(time.time() - t0, 1),
                          "gpu_name": (torch.cuda.get_device_name(0)
                                       if torch.cuda.is_available() else "cpu")},
           "ablation": {"ablation_group": "component_and_preprocessing_ablation",
                        "ablation_role": "variant", "baseline_model_id": "M22_v2",
                        "variable_changed": desc}}
    # Save the checkpoint. M44's attribution measures can then be re-run on any ablation
    # variant — which is the whole point of the P4 padding row: does removing the tiling
    # also remove the position-keying artefact? Without the weights that cannot be asked.
    torch.save({"epoch": int(best_ep), "best_score": float(best),
                "model_state": model.state_dict(), "cfg": cfg, "row": row_id},
               os.path.join(HERE, f"best_M45_{row_id}.pth"))
    json.dump(doc, open(os.path.join(HERE, f"results_M45_{row_id}.json"), "w"), indent=2)
    np.save(os.path.join(HERE, f"preds_M45_{row_id}.npy"),
            {"y_true": yte, "y_pred": best_pred, "patient_id": pid}, allow_pickle=True)
    print(f"    -> official {sc:.4f} {doc['best_metrics']['icbhi_score_official_ci95']}  "
          f"Se {se:.4f} Sp {sp:.4f}  ({time.time()-t0:.0f}s)")
    return doc


def summarise():
    rows = []
    for rid, (path, desc) in REUSED.items():
        if os.path.exists(path):
            d = json.load(open(path, encoding="utf-8")); bm = d["best_metrics"]
            rows.append((rid, desc, bm["icbhi_score_official"],
                         bm.get("icbhi_score_official_ci95"), bm["icbhi_se_official"],
                         bm["icbhi_sp_official"], bm["f1_macro"]))
    for f in sorted(glob.glob(os.path.join(HERE, "results_M45_*.json"))):
        d = json.load(open(f, encoding="utf-8")); bm = d["best_metrics"]
        rows.append((d["meta"]["row"], d["meta"]["variable_changed"],
                     bm["icbhi_score_official"], bm["icbhi_score_official_ci95"],
                     bm["icbhi_se_official"], bm["icbhi_sp_official"], bm["f1_macro"]))
    rows.sort(key=lambda r: r[0])
    base = next((r[2] for r in rows if r[0] == "A0"), None)

    print("\n" + "=" * 100)
    print("  M45 — ABLATION TABLE (baseline A0 = M22_v2, corrected official split)")
    print("=" * 100)
    print(f"  {'row':4s} {'variable changed':44s} {'official':>8s} {'delta':>8s} "
          f"{'Se':>6s} {'Sp':>6s} {'F1':>6s}")
    print("  " + "-" * 96)
    for rid, desc, sc, ci, se, sp, f1 in rows:
        dl = f"{sc-base:+.4f}" if base is not None and rid != "A0" else "—"
        mark = " (add)" if rid in ADD_ROWS else ""
        print(f"  {rid:4s} {desc[:44]:44s} {sc:8.4f} {dl:>8s} {se:6.3f} {sp:6.3f} {f1:6.3f}{mark}")
    print("  " + "-" * 96)
    for rid, why in NOT_APPLICABLE.items():
        print(f"  {rid:4s} NOT APPLICABLE — {why}")
    print("=" * 100)
    json.dump({"baseline": "A0 (M22_v2)", "split": "official_60_40_patient_independent_corrected",
               "rows": [{"row": r[0], "variable_changed": r[1], "icbhi_score_official": r[2],
                         "ci95": r[3], "se": r[4], "sp": r[5], "f1_macro": r[6],
                         "delta_vs_A0": (round(r[2]-base, 4) if base and r[0] != "A0" else None),
                         "direction": ("add" if r[0] in ADD_ROWS else "remove")}
                        for r in rows],
               "row_direction_note": "P1-P3 ADD a stage that BASE does not have; every other "
                                     "row REMOVES or replaces one. Their deltas read in "
                                     "opposite directions.",
               "not_applicable": NOT_APPLICABLE},
              open(os.path.join(HERE, "M45_ablation_table.json"), "w"), indent=2)
    print(f"\n  wrote M45_ablation_table.json")


def selftest():
    """Prove the three new stages do what their names say, without touching ICBHI.

    A preprocessing stage that silently does nothing would show up as a delta of ~0 and be
    reported as "the stage does not help", which is the wrong conclusion from a no-op.
    """
    ok = True
    t = np.arange(int(SR * 1.0)) / SR

    lo = np.sin(2 * np.pi * 500 * t).astype(np.float32)      # inside the band
    hi = np.sin(2 * np.pi * 6000 * t).astype(np.float32)     # outside it
    keep = float(np.std(butter_bandpass(lo)) / np.std(lo))
    kill = float(np.std(butter_bandpass(hi)) / np.std(hi))
    print(f"  band-pass    500 Hz kept {keep:.3f} (want ~1) | 6 kHz kept {kill:.4f} (want ~0)")
    ok &= keep > 0.9 and kill < 0.05

    quiet = (lo * 0.01).astype(np.float32)
    peak = float(np.max(np.abs(peak_normalise(quiet))))
    print(f"  peak norm    max |a| = {peak:.4f} (want 1.0)")
    ok &= abs(peak - 1.0) < 1e-3

    # Exponential, not squared-Gaussian: that is the distribution of power in a
    # stationary STFT bin, and the percentile the gate keys on depends on which it is.
    rng = np.random.default_rng(0)
    noise = rng.exponential(1.0, (64, 100))                  # stationary noise floor
    noise[:, 50] += 400.0                                    # one transient frame
    gated = spectral_gate(noise)
    floor_left = float(gated[:, :50].mean() / noise[:, :50].mean())
    transient_left = float(gated[:, 50].mean() / noise[:, 50].mean())
    print(f"  spectral gate  floor kept {floor_left:.3f} (want <0.6) | "
          f"transient kept {transient_left:.3f} (want >0.9)")
    ok &= floor_left < 0.6 and transient_left > 0.9

    keys = set()
    for r in ("A2", "P1", "P2", "P3", "P4"):
        cfg = dict(BASE); cfg.update({k: v for k, v in ROWS[r].items() if not k.startswith("_")})
        keys.add((f"{cfg['n_mels']}m_{cfg['duration_s']}s_{cfg['padding']}_{int(cfg['minmax'])}"
                  f"_bp{int(cfg['bandpass'])}_dn{int(cfg['denoise'])}_an{int(cfg['ampnorm'])}"))
    print(f"  cache keys   {len(keys)} distinct across A2/P1/P2/P3/P4 (want 5)")
    ok &= len(keys) == 5

    print("\n  SELFTEST", "PASS" if ok else "FAIL")
    return 0 if ok else 1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--row", choices=list(ROWS))
    ap.add_argument("--all", action="store_true")
    ap.add_argument("--summarise", action="store_true")
    ap.add_argument("--audio_dir", default=os.environ.get(
        "ICBHI_AUDIO_DIR", r"C:\Users\Barshon\Desktop\ICBHI_final_database"))
    ap.add_argument("--split_file",
                    default=os.path.join(REPO, "Asif's", "ICBHI_challenge_train_test.txt"))
    args = ap.parse_args()

    if args.selftest:
        return selftest()
    if args.summarise:
        return summarise()
    rows = corrected_split_index(args.audio_dir, args.split_file)
    todo = ([r for r in ROWS if not os.path.exists(os.path.join(HERE, f"results_M45_{r}.json"))]
            if args.all else [args.row])
    print(f"  test cycles {sum(1 for r in rows if r['split']=='test')} | rows to run: {todo}")
    for r in todo:
        run_row(r, rows, args)
    summarise()


if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/m45_ablation.py', 'w', encoding='utf-8') as fh:
    fh.write(M45_ABLATION_SRC)
print('wrote m45_ablation.py', len(M45_ABLATION_SRC), 'chars')


In [ ]:
M49_XVAL_SRC = r'''#!/usr/bin/env python3
"""
M49 - external validation of the best ICBHI model on SPRSound (2022) and HF_Lung_V1.

WHAT THIS IS
    Inference only. The checkpoint trained on the corrected ICBHI official split is run,
    unchanged and unadapted, over two corpora it has never seen, and scored with the same
    official ICBHI metric. No fine-tuning, no threshold tuning, no target label used to fit
    anything in the headline block. It is the "does this transfer" number the paper lacks.

WHY IT IMPORTS m45_ablation INSTEAD OF RE-IMPLEMENTING THE PIPELINE
    `log_mel`, `official`, `LABEL_OF` and the corrected-split loader come from
    `Asif's/M45/m45_ablation.py` verbatim - the module that trained the checkpoint. A second
    implementation of the mel parameters would turn every cross-dataset delta into a
    measurement of the gap between two scripts. The one thing that IS copied is the `Net`
    class, because it is defined inside `run_row` and cannot be imported; `load_state_dict`
    runs with strict=True so a structural drift raises rather than silently half-loading.

THE CHECK THAT MAKES THE REST BELIEVABLE
    `verify_on_icbhi()` re-scores the checkpoint on the 2,636 ICBHI test cycles and asserts
    the result equals the score stored inside the checkpoint (P3: 0.5764, M22_v2: 0.5602).
    That proves this file's preprocessing, channel expansion and ImageNet normalisation are
    the training run's, so a low external score reads as domain shift rather than as a bug
    in the harness. It runs BEFORE any external number is computed, and a mismatch aborts.

TAXONOMY MAPPING - THE ONE PLACE THIS EVALUATION CAN GO WRONG QUIETLY
    Neither corpus uses ICBHI's four labels, so both are mapped. Every mapping is spelled
    out below and copied into the results JSON, because a reader cannot check a
    cross-dataset score without it. An unknown label string RAISES with the offending
    vocabulary listed - it is never bucketed into Normal, which would inflate Sp.

    SPRSound event -> ICBHI        HF_Lung_V1 -> ICBHI
      Normal              Normal     I / E             phase, builds the cycle
      Fine/Coarse Crackle Crackle    D                 Crackle bit
      Wheeze / Rhonchi /  Wheeze     Wheeze / Stridor  Wheeze bit
        Stridor    (= CAS)             / Rhonchi (= CAS)
      Wheeze+Crackle      Both       both bits set     Both

    Rhonchi and stridor go to Wheeze because both source taxonomies define them as
    continuous adventitious sounds (CAS), which is the class ICBHI calls Wheeze;
    HF_Lung_V1's own paper pools W, S and R into CAS for exactly this reason.

UNIT OF ANALYSIS
    ICBHI scores respiratory cycles. SPRSound's event annotations are cycle-like segments
    and are the headline unit. HF_Lung_V1 annotates breath phases, so an inhalation and the
    exhalation that follows it are paired into one cycle - the ICBHI definition. SPRSound
    record-level labels are a secondary row on a DIFFERENT unit (a whole ~9 s recording
    truncated to the model's 8 s window) and are not comparable to the event-level number.

CONFIDENCE INTERVALS
    SPRSound filenames carry a patient ID, so its CI is a patient-level bootstrap, the same
    estimator the paper uses. HF_Lung_V1 filenames carry a timestamp and no patient ID, so
    its CI is a RECORDING-level bootstrap and is reported under that name. Calling it
    patient-level would overstate what the resampling controls for.

USAGE
    python m49_xval.py --selftest                       # synthetic corpora, no real data
    python m49_xval.py --dataset sprsound --root .../BioCAS2022 --ckpt .../best_M45_P3.pth
    python m49_xval.py --dataset hflung   --root .../HF_Lung_V1 --ckpt .../best_M45_P3.pth
"""
from __future__ import annotations

import argparse
import datetime
import glob
import json
import os
import platform
import re
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))

# m45_ablation is the module that trained the checkpoint. On Kaggle the notebook writes it
# next to this file; locally it lives in the repo. Both are tried, and failing to find it is
# fatal - a fallback re-implementation is the exact drift this import exists to prevent.
for _cand in (HERE, os.path.abspath(os.path.join(HERE, "..", "Asif's", "M45"))):
    if os.path.exists(os.path.join(_cand, "m45_ablation.py")) and _cand not in sys.path:
        sys.path.insert(0, _cand)
import m45_ablation as M45          # noqa: E402

CLASSES = M45.CLASSES               # ["Normal", "Crackle", "Wheeze", "Both"]
SR = M45.SR                         # 16000

# The committed P3 confusion matrix, used by the self-test to prove the metric being applied
# is the official one and not the inflated macro variant the project shipped before the audit.
P3_ICBHI_CM = [[1175, 231, 112, 42], [277, 308, 14, 18],
               [189, 40, 113, 31], [40, 15, 22, 9]]
P3_ICBHI_SCORE = 0.5764


# ================================================================== label taxonomies
def _norm(s):
    """Collapse whitespace and case so 'CAS & DAS' and 'cas  &  das' are one key."""
    return re.sub(r"\s+", " ", str(s)).strip().lower()


SPR_EVENT_TO_LABEL = {
    "normal": 0,
    "fine crackle": 1, "coarse crackle": 1, "crackle": 1,
    "wheeze": 2, "rhonchi": 2, "stridor": 2,
    "wheeze+crackle": 3, "wheeze + crackle": 3, "wheeze & crackle": 3,
    "wheeze and crackle": 3, "wheeze&crackle": 3,
}
SPR_RECORD_TO_LABEL = {"normal": 0, "das": 1, "cas": 2, "cas & das": 3, "cas&das": 3}
SPR_RECORD_EXCLUDE = {"poor quality"}   # unusable audio by the annotators' own judgement

HFL_PHASE = {"I", "E"}                          # inhalation / exhalation - build the cycle
HFL_CRACKLE = {"D"}                             # discontinuous adventitious sound
HFL_WHEEZE = {"Wheeze", "Stridor", "Rhonchi"}   # continuous adventitious sound (CAS)
HFL_KNOWN = HFL_PHASE | HFL_CRACKLE | HFL_WHEEZE

MAPPING_DOC = {
    "sprsound_event": {k: CLASSES[v] for k, v in SPR_EVENT_TO_LABEL.items()},
    "sprsound_record": {k: CLASSES[v] for k, v in SPR_RECORD_TO_LABEL.items()},
    "sprsound_record_excluded": sorted(SPR_RECORD_EXCLUDE),
    "hflung_phase_tokens": sorted(HFL_PHASE),
    "hflung_crackle_tokens": sorted(HFL_CRACKLE),
    "hflung_wheeze_tokens": sorted(HFL_WHEEZE),
    "rationale": ("Rhonchi and stridor are continuous adventitious sounds in both source "
                  "taxonomies and map to ICBHI's Wheeze class; HF_Lung_V1's own paper pools "
                  "W/S/R into CAS."),
}


# ================================================================== SPRSound
def _sf_info(path):
    """Header-only read: duration and native sample rate, without decoding the audio."""
    import soundfile as sf
    i = sf.info(path)
    return float(i.duration), int(i.samplerate)


def sprsound_index(root, level="event", min_event_s=0.05, verbose=True):
    """Index SPRSound into M45-shaped rows: wav / start / end / label / patient_id.

    `root` should be the BioCAS2022 directory (the 2022 release). The walk is recursive and
    pairing is by filename stem, not by directory, because test2022_json/ nests
    intra_test_json/ and inter_test_json/ while test2022_wav/ is flat.
    """
    wavs, jsons = {}, {}
    for p in glob.glob(os.path.join(root, "**", "*.wav"), recursive=True):
        wavs[os.path.splitext(os.path.basename(p))[0]] = p
    for p in glob.glob(os.path.join(root, "**", "*.json"), recursive=True):
        jsons[os.path.splitext(os.path.basename(p))[0]] = p
    paired = sorted(set(wavs) & set(jsons))
    if not paired:
        raise FileNotFoundError(
            f"no wav/json pairs under {root!r}. Point --root at the BioCAS2022 directory "
            "(it must contain train2022_wav/ and train2022_json/).")

    rows, seen_types = [], {}
    dropped = {"short_or_reversed": 0, "past_eof": 0, "excluded_record_label": 0,
               "no_annotation": 0}
    for stem in paired:
        wav = wavs[stem]
        try:
            doc = json.load(open(jsons[stem], encoding="utf-8"))
        except Exception as e:
            raise RuntimeError(f"unreadable annotation {jsons[stem]}") from e
        dur, native_sr = _sf_info(wav)
        pid = stem.split("_")[0]        # <patient>_<age>_<gender>_<location>_<record no>
        jp = jsons[stem].replace("\\", "/")
        subset = ("inter_test" if "inter_test" in jp else
                  "intra_test" if "intra_test" in jp else
                  "test" if "/test" in jp else "train")

        if level == "record":
            raw = _norm(doc.get("record_annotation", ""))
            seen_types[raw] = seen_types.get(raw, 0) + 1
            if raw in SPR_RECORD_EXCLUDE:
                dropped["excluded_record_label"] += 1
                continue
            if raw not in SPR_RECORD_TO_LABEL:
                if not raw:
                    dropped["no_annotation"] += 1
                    continue
                raise ValueError(f"unmapped SPRSound record label {raw!r} in {stem}. "
                                 f"Known: {sorted(SPR_RECORD_TO_LABEL)}")
            rows.append({"wav": wav, "stem": stem, "patient_id": pid, "subset": subset,
                         "start": 0.0, "end": dur, "label": SPR_RECORD_TO_LABEL[raw],
                         "native_sr": native_sr})
            continue

        events = doc.get("event_annotation") or []
        if not events:
            dropped["no_annotation"] += 1
            continue
        for ev in events:
            raw = _norm(ev.get("type", ""))
            seen_types[raw] = seen_types.get(raw, 0) + 1
            if raw not in SPR_EVENT_TO_LABEL:
                raise ValueError(
                    f"unmapped SPRSound event label {raw!r} in {stem}. "
                    f"Known: {sorted(SPR_EVENT_TO_LABEL)}. Bucketing an unknown class into "
                    "Normal would silently inflate specificity.")
            s, e = float(ev["start"]) / 1000.0, float(ev["end"]) / 1000.0   # ms -> s
            if e - s < min_event_s:
                dropped["short_or_reversed"] += 1
                continue
            if s >= dur:
                dropped["past_eof"] += 1
                continue
            rows.append({"wav": wav, "stem": stem, "patient_id": pid, "subset": subset,
                         "start": s, "end": min(e, dur),
                         "label": SPR_EVENT_TO_LABEL[raw], "native_sr": native_sr})

    if verbose:
        print(f"  SPRSound[{level}] {len(rows)} rows from {len(paired)} recordings, "
              f"{len({r['patient_id'] for r in rows})} patients")
        print(f"    label vocabulary seen: {dict(sorted(seen_types.items()))}")
        print(f"    dropped: {dropped}")
    return rows, {"recordings_paired": len(paired), "label_vocabulary": seen_types,
                  "dropped": dropped}


# ================================================================== HF_Lung_V1
def _hfl_time(s):
    """Seconds from '1.5' or from 'HH:MM:SS.mmm'. Both forms appear in the wild."""
    if ":" in s:
        h, m, sec = s.split(":")
        return int(h) * 3600 + int(m) * 60 + float(sec)
    return float(s)


def _overlap(a1, a2, b1, b2):
    return max(0.0, min(a2, b2) - max(a1, b1))


def hflung_group(stem):
    """Cluster the segments cut from one original recording.

    HF_Lung_V1 has no patient ID in the filename. `trunc_yyyy-mm-dd-HH-MM-ss-LX_N` is the
    Nth 15 s slice at auscultation site LX of ONE session, so all of its slices must land on
    the same side of a bootstrap resample or the interval comes out optimistic.
    """
    m = re.match(r"^(trunc_\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2})", stem)
    return m.group(1) if m else stem


def hflung_parse_label(path):
    """`<label> <start> <end>`, whitespace-delimited, one event per line."""
    out = []
    for ln, raw in enumerate(open(path, encoding="utf-8", errors="replace"), 1):
        p = raw.split()
        if not p:
            continue
        if len(p) < 3:
            raise ValueError(f"{path}:{ln}: expected '<label> <start> <end>', got {raw!r}")
        out.append((p[0], _hfl_time(p[1]), _hfl_time(p[2])))
    return out


def hflung_cycles(events, pair_gap_s=1.0):
    """Pair each inhalation with the exhalation that follows it into one ICBHI-style cycle.

    Returns (start, end, phases) where `phases` is "I+E", "I" or "E".

    An unpaired phase becomes a cycle on its own rather than being discarded: dropping the
    lone phases would throw away the breaths at the clip boundaries, which is exactly where a
    15 s window cuts a cycle in half, and that loss is not random with respect to the label.

    It is also not a rare edge case. HF_Lung_V1 carries 34,095 inhalation labels against
    18,349 exhalation labels, so most inhalations have no annotated exhalation to pair with
    and the majority of rows end up being single phases rather than whole cycles. That is a
    real difference from ICBHI's unit and it is counted into `index_stats.phase_composition`
    instead of being smoothed over - a row that is one inhalation is roughly half the duration
    of the ICBHI cycle the model was trained on.
    """
    ph = sorted([e for e in events if e[0] in HFL_PHASE], key=lambda e: (e[1], e[2]))
    out, i = [], 0
    while i < len(ph):
        lab, s, e = ph[i]
        if (i + 1 < len(ph) and lab == "I" and ph[i + 1][0] == "E"
                and ph[i + 1][1] - e <= pair_gap_s):
            out.append((s, ph[i + 1][2], "I+E"))
            i += 2
        else:
            out.append((s, e, lab))
            i += 1
    return out


def hflung_index(root, min_overlap_s=0.05, pair_gap_s=1.0, min_cycle_s=0.05, verbose=True):
    """Index HF_Lung_V1 into M45-shaped rows. `root` holds the extracted train/ and test/."""
    wavs = {}
    for p in glob.glob(os.path.join(root, "**", "*.wav"), recursive=True):
        wavs[os.path.splitext(os.path.basename(p))[0]] = p
    if not wavs:
        raise FileNotFoundError(f"no .wav under {root!r}. Extract train.7z / test.7z first.")

    rows, seen_types = [], {}
    phases = {"I+E": 0, "I": 0, "E": 0}
    dropped = {"no_label_file": 0, "no_phase_labels": 0, "short_or_reversed": 0,
               "past_eof": 0}
    for stem in sorted(wavs):
        wav = wavs[stem]
        lab = os.path.join(os.path.dirname(wav), stem + "_label.txt")
        if not os.path.exists(lab):
            dropped["no_label_file"] += 1
            continue
        events = hflung_parse_label(lab)
        for t, _, _ in events:
            seen_types[t] = seen_types.get(t, 0) + 1
        unknown = {t for t, _, _ in events} - HFL_KNOWN
        if unknown:
            raise ValueError(f"unmapped HF_Lung_V1 label(s) {sorted(unknown)} in {lab}. "
                             f"Known: {sorted(HFL_KNOWN)}.")

        cycles = hflung_cycles(events, pair_gap_s)
        if not cycles:
            dropped["no_phase_labels"] += 1
            continue
        adv = [e for e in events if e[0] not in HFL_PHASE]
        dur, native_sr = _sf_info(wav)
        wp = wav.replace("\\", "/")
        subset = "test" if "/test/" in wp else "train"
        for s, e, ph in cycles:
            if e - s < min_cycle_s:
                dropped["short_or_reversed"] += 1
                continue
            if s >= dur:
                dropped["past_eof"] += 1
                continue
            crk = any(_overlap(s, e, a, b) >= min_overlap_s
                      for t, a, b in adv if t in HFL_CRACKLE)
            whz = any(_overlap(s, e, a, b) >= min_overlap_s
                      for t, a, b in adv if t in HFL_WHEEZE)
            phases[ph] += 1
            rows.append({"wav": wav, "stem": stem, "patient_id": hflung_group(stem),
                         "subset": subset, "start": s, "end": min(e, dur), "phases": ph,
                         "label": M45.LABEL_OF[(int(crk), int(whz))],
                         "native_sr": native_sr})

    if verbose:
        print(f"  HF_Lung_V1 {len(rows)} cycles from {len(wavs)} recordings, "
              f"{len({r['patient_id'] for r in rows})} recording groups")
        print(f"    label vocabulary seen: {dict(sorted(seen_types.items()))}")
        print(f"    phase composition: {phases}  "
              f"({100.0 * phases['I+E'] / max(len(rows), 1):.1f}% are whole I+E cycles)")
        print(f"    dropped: {dropped}")
    return rows, {"recordings_seen": len(wavs), "label_vocabulary": seen_types,
                  "phase_composition": phases,
                  "phase_composition_note": (
                      "HF_Lung_V1 annotates far more inhalations than exhalations, so most "
                      "rows are a single phase rather than a whole I+E cycle. A single-phase "
                      "row is roughly half the duration of the ICBHI cycle the model was "
                      "trained on; see dataset_info.segment_stats for the tiling this "
                      "implies."),
                  "dropped": dropped}


# ================================================================== model
def build_model(cfg, device):
    """The M22_v2 / M45 network, copied verbatim from `m45_ablation.run_row`.

    It cannot be imported (it is a closure over `cfg` inside `run_row`), so it is copied -
    and `load_checkpoint` uses strict=True so any structural drift raises instead of quietly
    loading a subset of the weights. `embed` is the only addition: it is the forward pass
    split at the penultimate layer so the frozen-feature probe can reuse it.
    """
    import torch, torch.nn as nn, torchvision

    mean = torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)
    std = torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.features = torchvision.models.mobilenet_v2(weights=None).features
            s.gap = nn.AdaptiveAvgPool2d((1, 1))
            s.dropout = nn.Dropout(cfg["dropout"])
            s.classifier = nn.Linear(1280, 4)
            s.norm = cfg["pretrained"]

        def embed(s, x):
            x = x.repeat(1, 3, 1, 1)
            if s.norm:
                x = (x - mean.to(x.device)) / std.to(x.device)
            return s.gap(s.features(x)).flatten(1)

        def forward(s, x):
            return s.classifier(s.dropout(s.embed(x)))

    return Net().to(device)


def load_checkpoint(path, device=None):
    """Return (model, cfg, meta). cfg = M45.BASE overlaid with the checkpoint's own cfg.

    The overlay is what lets one loader serve both checkpoints: M45's cfg names every
    preprocessing flag, M22_v2's predates three of them, and BASE holds the values those
    flags had when M22_v2 was trained (wrap padding, min-max on, no amplitude normalisation).
    """
    import torch
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if "model_state" not in ck:
        raise ValueError(f"{path} has keys {list(ck)} - expected a dict with 'model_state'")
    cfg = dict(M45.BASE)
    cfg.update(ck.get("cfg", {}))
    model = build_model(cfg, device)
    model.load_state_dict(ck["model_state"], strict=True)
    model.eval()
    # Counted from the state_dict tensors, buffers included - the convention
    # `Asif's/audit/backfill_efficiency.py` used for every other results file in the repo
    # (2,263,160 for this architecture, not the 2,228,996 that `parameters()` returns).
    meta = {"path": os.path.basename(path), "row": ck.get("row"), "epoch": ck.get("epoch"),
            "reported_icbhi_score": ck.get("best_score"),
            "total_params": int(sum(int(v.numel()) for v in ck["model_state"].values()
                                    if hasattr(v, "numel")))}
    return model, cfg, meta


def predict(model, rows, cfg, device=None, batch_size=64, num_workers=None,
            want_features=False, verbose=True):
    """One forward pass over `rows`.

    Spectrograms are computed on the fly and not cached: a cross-dataset run is a single
    pass, so an M45-style disk cache would cost 2-5 GB of Kaggle working storage and buy
    nothing back.
    """
    import torch
    from torch.utils.data import Dataset, DataLoader

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    if num_workers is None:
        num_workers = 0 if os.name == "nt" else 2

    class DS(Dataset):
        def __len__(s):
            return len(rows)

        def __getitem__(s, i):
            r = rows[i]
            return torch.from_numpy(M45.log_mel(r["wav"], r["start"], r["end"], cfg))

    dl = DataLoader(DS(), batch_size=batch_size, shuffle=False, num_workers=num_workers)
    logits, feats, t0 = [], [], time.time()
    with torch.no_grad():
        for bi, x in enumerate(dl):
            x = x.to(device)
            f = model.embed(x)
            logits.append(model.classifier(f).float().cpu().numpy())
            if want_features:
                feats.append(f.float().cpu().numpy())
            if verbose and (bi + 1) % 50 == 0:
                done = min((bi + 1) * batch_size, len(rows))
                print(f"      {done}/{len(rows)}  ({time.time() - t0:.0f}s)")
    out = {"logits": np.concatenate(logits), "seconds": time.time() - t0}
    if want_features:
        out["features"] = np.concatenate(feats)
    return out


# ================================================================== metrics
def _f(x, nd=4):
    """Round, or None if the value is not finite.

    `official()` returns NaN when a subset contains no Normal or no abnormal rows, and
    `json.dump` would happily write that as a bare `NaN`, which is not valid JSON and which
    every downstream reader would then either reject or misparse. A missing number has to
    look missing.
    """
    x = float(x)
    return round(x, nd) if np.isfinite(x) else None


def group_bootstrap_ci(y, pred, groups, n_boot=2000, seed=42):
    """M45.patient_ci with the grouping unit named by the caller instead of assumed.

    A resample that happens to draw only Normal groups scores NaN and is skipped. If EVERY
    resample degenerates - which is what a `--limit` smoke run on a one-class slice does -
    the interval is `[None, None]` rather than a crash or, worse, a fabricated interval.
    `degenerate_resample_fraction` in the caller's block says how often it happened.
    """
    from sklearn.metrics import confusion_matrix
    y, pred, groups = np.asarray(y), np.asarray(pred), np.asarray(groups)
    uq = np.unique(groups)
    ix = {g: np.flatnonzero(groups == g) for g in uq}
    rng = np.random.default_rng(seed)
    v = []
    for _ in range(n_boot):
        s = np.concatenate([ix[g] for g in rng.choice(uq, len(uq), replace=True)])
        x = M45.official(confusion_matrix(y[s], pred[s], labels=[0, 1, 2, 3]))[0]
        if np.isfinite(x):
            v.append(x)
    if not v:
        return [None, None]
    return [round(float(z), 4) for z in np.percentile(v, [2.5, 97.5])]


def binary_rescore(cm):
    """Rescore the SAME predictions as detect-only (any abnormal vs Normal).

    Sp is unchanged by construction; Se rises because a crackle called a wheeze now counts.
    The gap is the cost of sub-typing, the quantity the paper's error table reports
    in-domain, so the external and in-domain versions are directly comparable.
    """
    cm = np.asarray(cm, float)
    sp = cm[0, 0] / cm[0].sum() if cm[0].sum() else float("nan")
    abn = cm[1:].sum()
    se = cm[1:, 1:].sum() / abn if abn else float("nan")
    return {"se": _f(se), "sp": _f(sp), "icbhi_score_official": _f((se + sp) / 2)}


def score_block(y, pred, groups, group_kind, n_boot=2000):
    from sklearn.metrics import (confusion_matrix, accuracy_score, f1_score,
                                 precision_recall_fscore_support)
    cm = confusion_matrix(y, pred, labels=[0, 1, 2, 3])
    sc, se, sp = M45.official(cm)
    p, r, f, sup = precision_recall_fscore_support(y, pred, labels=[0, 1, 2, 3],
                                                   zero_division=0)
    rown = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    return {
        "icbhi_score_official": _f(sc),
        "icbhi_score_official_ci95": group_bootstrap_ci(y, pred, groups, n_boot),
        "icbhi_score_official_ci95_unit": group_kind,
        "icbhi_se_official": _f(se),
        "icbhi_sp_official": _f(sp),
        "accuracy": round(float(accuracy_score(y, pred)), 4),
        "precision_macro": round(float(p.mean()), 4),
        "recall_macro": round(float(r.mean()), 4),
        "f1_macro": round(float(f1_score(y, pred, average="macro", zero_division=0)), 4),
        "per_class": {CLASSES[i]: {"precision": round(float(p[i]), 4),
                                   "recall": round(float(r[i]), 4),
                                   "f1": round(float(f[i]), 4),
                                   "support": int(sup[i]),
                                   "predicted": int(cm[:, i].sum())} for i in range(4)},
        "confusion_matrix_raw": cm.tolist(),
        "confusion_matrix_normalized": np.round(rown, 4).tolist(),
        "label_distribution_true": np.bincount(np.asarray(y), minlength=4).tolist(),
        "label_distribution_pred": np.bincount(np.asarray(pred), minlength=4).tolist(),
        "binary_detection": binary_rescore(cm),
    }


# ================================================================== the ICBHI gate
def verify_on_icbhi(model, cfg, ckpt_meta, audio_dir, split_file, tol=1e-3, **kw):
    """Re-score the checkpoint on ICBHI and require it to reproduce its own stored score.

    This is the gate. An external score produced by an unverified forward path is not a
    result, so a mismatch raises rather than warning.
    """
    from sklearn.metrics import confusion_matrix
    rows = M45.corrected_split_index(audio_dir, split_file)
    te = [r for r in rows if r["split"] == "test"]
    print(f"  ICBHI verification: {len(te)} test cycles, "
          f"{len({r['patient_id'] for r in te})} patients")
    y = np.array([r["label"] for r in te])
    pred = predict(model, te, cfg, **kw)["logits"].argmax(1)
    got = M45.official(confusion_matrix(y, pred, labels=[0, 1, 2, 3]))[0]
    exp = ckpt_meta.get("reported_icbhi_score")
    print(f"  reproduced {got:.4f} | checkpoint reports {exp:.4f} | tol {tol}")
    if exp is None or abs(got - exp) > tol:
        raise AssertionError(
            f"ICBHI reproduction failed: {got:.4f} vs {exp}. The forward path here does not "
            "match the training run - do NOT report any external number from it.")
    print("  PASS - forward path reproduces the training run.")
    return round(float(got), 4)


# ================================================================== optional probe
def feature_probe(features, y, groups, seed=42, n_splits=5):
    """Frozen-feature linear probe, grouped CV, every clip out-of-sample.

    Separates two failure modes one zero-shot number cannot tell apart: the representation
    carries nothing about this corpus, versus the representation is fine and only the
    ICBHI-fitted decision boundary fails to transfer. It uses target labels, so it is NOT a
    zero-shot number and lives in its own block.
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedGroupKFold
    from sklearn.metrics import confusion_matrix
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    y = np.asarray(y)
    oof = np.full(len(y), -1)
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr, te in skf.split(features, y, groups):
        # The scaler is inside the pipeline, so it is fitted on the training fold only and
        # the held-out fold never contributes to its statistics. Post-GAP activations are
        # all non-negative and vary by an order of magnitude across the 1,280 units, which
        # lbfgs converges on slowly and unevenly without it.
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0))
        clf.fit(features[tr], y[tr])
        oof[te] = clf.predict(features[te])
    assert (oof >= 0).all(), "grouped CV left clips unassigned"
    cm = confusion_matrix(y, oof, labels=[0, 1, 2, 3])
    sc, se, sp = M45.official(cm)
    # SPRSound's Both class is thin (34 events in the full 2022 release, 4 in the test half),
    # and below n_splits the folds cannot be stratified on it - sklearn warns to stderr,
    # which nobody reads off a finished notebook. Put the number in the result instead.
    support = np.bincount(y, minlength=4)
    return {"icbhi_score_official": _f(sc),
            "icbhi_score_official_ci95": group_bootstrap_ci(y, oof, groups),
            "icbhi_se_official": _f(se), "icbhi_sp_official": _f(sp),
            "confusion_matrix_raw": cm.tolist(),
            "class_support": {CLASSES[i]: int(support[i]) for i in range(4)},
            "min_class_support": int(support.min()),
            "stratification_ok": bool(support.min() >= n_splits),
            "note": (f"frozen-backbone logistic probe, grouped {n_splits}-fold CV. Uses "
                     "target labels - not a zero-shot result, and not comparable to the "
                     "headline number. If stratification_ok is false, the rarest class has "
                     "fewer members than folds and its per-fold estimate is unstable.")}


# ================================================================== reporting
def segment_stats(rows, cfg):
    """How hard the 8 s window has to work on this corpus.

    Every segment shorter than the window is tiled with `np.tile`, which leaves a
    discontinuity at each seam - the artefact M44 measured and the P4 ablation row tested.
    ICBHI's cycles have a median of 2.42 s and are tiled about 3.3x; a corpus of shorter
    segments is tiled harder, so the model sees proportionally more seam and less acoustics.
    That is a confound on any cross-dataset comparison and belongs in the results file rather
    than in someone's memory.
    """
    d = np.array([r["end"] - r["start"] for r in rows], float)
    win = float(cfg["duration_s"])
    return {"segment_seconds_median": round(float(np.median(d)), 3),
            "segment_seconds_p5_p95": [round(float(np.percentile(d, 5)), 3),
                                       round(float(np.percentile(d, 95)), 3)],
            "segment_seconds_min_max": [round(float(d.min()), 3), round(float(d.max()), 3)],
            "model_window_seconds": win,
            "tiling_factor_median": round(float(win / np.median(d)), 2),
            "fraction_longer_than_window": round(float((d >= win).mean()), 4),
            "icbhi_reference_median_seconds": 2.42,
            "note": ("segments shorter than the window are cyclically tiled; a larger "
                     "tiling factor than ICBHI's ~3.3x means proportionally more seam "
                     "artefact per input.")}


def make_results(model_id, dataset, level, rows, y, pred, cfg, ckpt_meta, index_stats,
                 icbhi_reference, seconds, extra=None):
    import torch
    groups = np.array([r["patient_id"] for r in rows])
    group_kind = "patient" if dataset == "SPRSound" else "recording_group"
    doc = {
        "meta": {
            "model_id": model_id,
            "model_name": f"external validation of {ckpt_meta['path']} on {dataset}",
            "contributor": "OWMTL team",
            "date_completed": datetime.datetime.now().strftime("%Y-%m-%d"),
            "is_augmented": False,
            "augmentation_method": "none (inference only)",
            "notes": ("Zero-shot cross-dataset evaluation. No fine-tuning, no threshold "
                      "tuning, no target label used to fit anything in best_metrics. The "
                      "checkpoint reproduced its own ICBHI score before this ran."),
        },
        "config": dict(cfg),
        "environment": {"platform": platform.platform(),
                        "python_version": platform.python_version(),
                        "pytorch_version": torch.__version__,
                        "gpu_name": (torch.cuda.get_device_name(0)
                                     if torch.cuda.is_available() else "cpu")},
        "dataset_info": {
            "dataset": dataset,
            "unit_of_analysis": level,
            "split_method": "external_validation_zero_shot_all_available_data",
            "train_samples": 0,
            "test_samples": int(len(y)),
            "test_groups": int(len(np.unique(groups))),
            "test_group_kind": group_kind,
            "subset_counts": {s: int(sum(1 for r in rows if r["subset"] == s))
                              for s in sorted({r["subset"] for r in rows})},
            "native_sample_rates_hz": sorted({int(r["native_sr"]) for r in rows}),
            "resampled_to_hz": SR,
            "index_stats": index_stats,
            "segment_stats": segment_stats(rows, cfg),
            "label_mapping": MAPPING_DOC,
        },
        "source_model": {
            "checkpoint": ckpt_meta["path"],
            "ablation_row": ckpt_meta.get("row"),
            "selected_epoch": ckpt_meta.get("epoch"),
            "icbhi_score_official_in_domain": icbhi_reference,
            "trained_on": "ICBHI_2017, official_60_40_patient_independent_corrected",
        },
        "best_epoch": {"epoch": ckpt_meta.get("epoch"),
                       "primary_metric": "icbhi_score_official",
                       "primary_metric_value": None},
        "best_metrics": score_block(y, pred, groups, group_kind),
        "efficiency": {
            "total_params": ckpt_meta.get("total_params"),
            "trainable_params": 0,
            "model_size_mb": (round(ckpt_meta["total_params"] * 4 / 1e6, 3)
                              if ckpt_meta.get("total_params") else None),
            "training_time_total_s": 0,
            "inference_time_total_s": round(seconds, 1),
            "inference_time_ms_per_sample": round(1000.0 * seconds / max(len(y), 1), 3),
            "gpu_name": (torch.cuda.get_device_name(0)
                         if torch.cuda.is_available() else "cpu"),
        },
        "ablation": {
            "ablation_group": "external_validation",
            "ablation_role": "variant",
            "baseline_model_id": ckpt_meta.get("row") or "M22_v2",
            "variable_changed": f"evaluation corpus: ICBHI_2017 -> {dataset}",
            "variables_held_constant": ["weights", "preprocessing", "official metric",
                                        "decision rule: argmax"],
            "component_flags": {"has_sound_event_head": True, "has_disease_head": False,
                                "has_cross_task_consistency": False,
                                "has_cqkd_regularization": False,
                                "has_openmax_rejection": False, "owl_stage": 0,
                                "compression_clusters": None},
            "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                             "consistency_weight": None},
        },
        "training_history": [],
    }
    ext = doc["best_metrics"]["icbhi_score_official"]
    doc["best_epoch"]["primary_metric_value"] = ext
    doc["transfer"] = {
        "icbhi_in_domain": icbhi_reference,
        "external": ext,
        "delta": (round(ext - icbhi_reference, 4) if ext is not None else None),
        "chance_reference": 0.5,
        "note": ("0.50 is what a constant always-Normal predictor scores on this metric, so "
                 "it is the floor an external number has to clear to mean anything."),
    }
    if extra:
        doc.update(extra)
    return doc


def plot_confusion(cm, title, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = np.asarray(cm, float)
    rown = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(4.4, 4.0))
    im = ax.imshow(rown, cmap="Blues", vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{rown[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if rown[i, j] > 0.5 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels(CLASSES, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title, fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)
    return path


def _s(x, sign=False):
    """Format a metric that may legitimately be absent (see `_f`)."""
    if x is None:
        return "  n/a "
    return f"{x:+.4f}" if sign else f"{x:.4f}"


def print_summary(doc):
    b, t, d = doc["best_metrics"], doc["transfer"], doc["dataset_info"]
    print("\n" + "=" * 78)
    print(f"  {doc['meta']['model_id']}  |  {d['dataset']}  |  unit: {d['unit_of_analysis']}")
    print("=" * 78)
    print(f"  n = {d['test_samples']} over {d['test_groups']} {d['test_group_kind']}s")
    print(f"  ICBHI official   {_s(b['icbhi_score_official'])} "
          f"{b['icbhi_score_official_ci95']}   "
          f"({b['icbhi_score_official_ci95_unit']}-level CI)")
    print(f"  Se {_s(b['icbhi_se_official'])}   Sp {_s(b['icbhi_sp_official'])}   "
          f"acc {b['accuracy']:.4f}   macro-F1 {b['f1_macro']:.4f}")
    print(f"  detect-only      {_s(b['binary_detection']['icbhi_score_official'])}  "
          f"(Se {_s(b['binary_detection']['se'])}, Sp unchanged)")
    print(f"  in-domain ICBHI  {_s(t['icbhi_in_domain'])}   ->   "
          f"delta {_s(t['delta'], sign=True)}   (0.50 = always-Normal)")
    ss = d.get("segment_stats")
    if ss:
        print(f"  segments         median {ss['segment_seconds_median']:.2f} s -> tiled "
              f"{ss['tiling_factor_median']:.2f}x into the {ss['model_window_seconds']:.0f} s "
              f"window (ICBHI median 2.42 s, ~3.3x)")
    print("  distribution over" + "".join(f"{c:>10s}" for c in CLASSES))
    print("    true         " + "".join(f"{v:10d}" for v in b["label_distribution_true"]))
    print("    predicted    " + "".join(f"{v:10d}" for v in b["label_distribution_pred"]))
    if "feature_probe" in doc:
        p = doc["feature_probe"]
        print(f"  frozen-feature probe (uses target labels): "
              f"{_s(p['icbhi_score_official'])} {p['icbhi_score_official_ci95']}")
    if b["icbhi_score_official"] is None:
        print("  !! the official score is undefined here - this slice has no Normal rows or "
              "no abnormal rows. Not a reportable result.")
    print("=" * 78)


# ================================================================== self-test
def _write_wav(path, seconds, sr, freq=440.0):
    import soundfile as sf
    t = np.arange(int(seconds * sr)) / sr
    sf.write(path, (0.2 * np.sin(2 * np.pi * freq * t)).astype(np.float32), sr)


def selftest(tmp=None):
    """Build synthetic SPRSound and HF_Lung_V1 corpora and run the whole path over them.

    Nothing here touches a real dataset or a GPU. It exists because the two expensive
    failure modes of this file are silent: a taxonomy mapping that buckets a class into the
    wrong bin, and a cycle builder that produces the right NUMBER of rows with the wrong
    times. Both would yield a plausible score that is simply not the quantity claimed.
    """
    import shutil
    import tempfile
    ok = True
    tmp = tmp or tempfile.mkdtemp(prefix="m49_selftest_")

    # -- the metric applied is the official one, not the inflated macro variant
    got = M45.official(P3_ICBHI_CM)[0]
    print(f"  metric        P3 matrix -> {got:.4f} (want {P3_ICBHI_SCORE})")
    ok &= abs(got - P3_ICBHI_SCORE) < 1e-4

    # -- SPRSound: one record per event class, plus a Poor Quality that must be excluded
    spr = os.path.join(tmp, "BioCAS2022", "train2022_wav")
    sprj = os.path.join(tmp, "BioCAS2022", "train2022_json")
    os.makedirs(spr, exist_ok=True)
    os.makedirs(sprj, exist_ok=True)
    cases = ["Normal", "Fine Crackle", "Coarse Crackle", "Wheeze", "Rhonchi", "Stridor",
             "Wheeze+Crackle"]
    for i, typ in enumerate(cases):
        stem = f"9000000{i}_5.0_0_p1_{100 + i}"
        _write_wav(os.path.join(spr, stem + ".wav"), 9.2, 8000)
        json.dump({"record_annotation": "Normal",
                   "event_annotation": [{"start": "1000", "end": "2500", "type": typ},
                                        {"start": "3000", "end": "4200", "type": typ}]},
                  open(os.path.join(sprj, stem + ".json"), "w"))
    stem = "90000099_5.0_0_p1_199"
    _write_wav(os.path.join(spr, stem + ".wav"), 9.2, 8000)
    json.dump({"record_annotation": "Poor Quality", "event_annotation": []},
              open(os.path.join(sprj, stem + ".json"), "w"))

    rows, st = sprsound_index(os.path.join(tmp, "BioCAS2022"), "event", verbose=False)
    counts = np.bincount([r["label"] for r in rows], minlength=4).tolist()
    print(f"  sprsound      event labels {counts} (want [2, 4, 6, 2])")
    ok &= counts == [2, 4, 6, 2]
    ok &= all(abs((r["end"] - r["start"]) - (1.5 if r["start"] == 1.0 else 1.2)) < 1e-9
              for r in rows)
    ok &= len({r["patient_id"] for r in rows}) == 7
    ok &= st["dropped"]["short_or_reversed"] == 0
    rrows, _ = sprsound_index(os.path.join(tmp, "BioCAS2022"), "record", verbose=False)
    print(f"  sprsound      record rows {len(rrows)} (want 7 - Poor Quality excluded)")
    ok &= len(rrows) == 7

    # -- HF_Lung_V1: one file per target class, built from I/E phases + adventitious spans
    hfl = os.path.join(tmp, "HF_Lung_V1", "train")
    os.makedirs(hfl, exist_ok=True)
    specs = {
        "steth_20190101_00_00_01": (["I 1.0 2.0", "E 2.1 3.5"], 0),
        "steth_20190101_00_00_02": (["I 1.0 2.0", "E 2.1 3.5", "D 1.5 1.6"], 1),
        "steth_20190101_00_00_03": (["I 1.0 2.0", "E 2.1 3.5", "Wheeze 2.5 3.0"], 2),
        "steth_20190101_00_00_04": (["I 1.0 2.0", "E 2.1 3.5", "Rhonchi 2.5 3.0"], 2),
        "steth_20190101_00_00_05": (["I 1.0 2.0", "E 2.1 3.5", "D 1.5 1.6",
                                     "Stridor 2.5 3.0"], 3),
        # an adventitious span outside the cycle must NOT label it
        "steth_20190101_00_00_06": (["I 1.0 2.0", "E 2.1 3.5", "D 9.0 9.2"], 0),
    }
    for stem, (lines, _) in specs.items():
        _write_wav(os.path.join(hfl, stem + ".wav"), 15.0, 4000)
        open(os.path.join(hfl, stem + "_label.txt"), "w").write("\n".join(lines) + "\n")
    for n in (1, 2):        # two slices of one session - they must share a bootstrap group
        stem = f"trunc_2019-01-01-00-00-07-L1_{n}"
        _write_wav(os.path.join(hfl, stem + ".wav"), 15.0, 4000)
        open(os.path.join(hfl, stem + "_label.txt"), "w").write("I 1.0 2.0\nE 2.1 3.5\n")

    hrows, hst = hflung_index(os.path.join(tmp, "HF_Lung_V1"), verbose=False)
    by_stem = {r["stem"]: r for r in hrows}
    print(f"  hflung        {len(hrows)} cycles from 8 files (want 8 - one cycle each)")
    ok &= len(hrows) == 8
    for stem, (_, want) in specs.items():
        gotl = by_stem[stem]["label"]
        ok &= gotl == want
        if gotl != want:
            print(f"    MISMATCH {stem}: got {CLASSES[gotl]} want {CLASSES[want]}")
    c = by_stem["steth_20190101_00_00_01"]
    print(f"  hflung        I+E paired into [{c['start']}, {c['end']}] as {c['phases']!r} "
          "(want [1.0, 3.5] 'I+E')")
    ok &= abs(c["start"] - 1.0) < 1e-9 and abs(c["end"] - 3.5) < 1e-9 and c["phases"] == "I+E"
    # an inhalation with no exhalation must survive as its own row, tagged as one phase -
    # HF_Lung_V1 has nearly twice as many I labels as E labels, so this is the common case
    lone = hflung_cycles([("I", 1.0, 2.0), ("I", 5.0, 6.0), ("E", 6.1, 7.0)])
    print(f"  hflung        unpaired I kept: {lone} (want I, then I+E)")
    ok &= [x[2] for x in lone] == ["I", "I+E"] and abs(lone[1][1] - 7.0) < 1e-9
    g = {r["stem"]: r["patient_id"] for r in hrows}
    same = g["trunc_2019-01-01-00-00-07-L1_1"] == g["trunc_2019-01-01-00-00-07-L1_2"]
    print(f"  hflung        two slices of one session share a group: {same} (want True)")
    ok &= same and hst["dropped"]["no_label_file"] == 0

    # -- an unknown label must raise, never be bucketed into Normal
    bad = os.path.join(tmp, "bad", "train")
    os.makedirs(bad, exist_ok=True)
    _write_wav(os.path.join(bad, "steth_20190101_00_00_09.wav"), 15.0, 4000)
    open(os.path.join(bad, "steth_20190101_00_00_09_label.txt"), "w").write("Squawk 1 2\n")
    try:
        hflung_index(os.path.join(tmp, "bad"), verbose=False)
        print("  unknown label did NOT raise (want ValueError)")
        ok = False
    except ValueError:
        print("  unknown label raises as required")

    # -- the tensor the model actually receives
    cfg = dict(M45.BASE)
    cfg["ampnorm"] = True
    x = M45.log_mel(rows[0]["wav"], rows[0]["start"], rows[0]["end"], cfg)
    print(f"  log_mel       shape {x.shape} range [{x.min():.2f}, {x.max():.2f}] "
          "(want (1, 128, 801) inside [0, 1])")
    ok &= x.shape == (1, 128, 801) and x.min() >= -1e-6 and x.max() <= 1 + 1e-6

    # -- forward path
    try:
        import torch
        m = build_model(cfg, "cpu")
        with torch.no_grad():
            o = m(torch.from_numpy(np.stack([x, x])))
        print(f"  forward       {tuple(o.shape)} (want (2, 4))")
        ok &= tuple(o.shape) == (2, 4)
    except ImportError as e:                                  # torch absent locally
        print(f"  forward       SKIPPED ({e})")

    # -- the estimators
    y = np.array([0] * 40 + [1] * 20 + [2] * 20 + [3] * 20)
    grp = np.repeat(np.arange(20), 5)
    lo, hi = group_bootstrap_ci(y, y.copy(), grp, n_boot=200)
    print(f"  bootstrap     perfect predictions -> [{lo}, {hi}] (want [1.0, 1.0])")
    ok &= lo == 1.0 and hi == 1.0
    # An all-Normal slice makes the official score undefined. It must come back missing,
    # not as a crash and not as a fabricated interval.
    z = np.zeros(10, int)
    deg = group_bootstrap_ci(z, z.copy(), np.arange(10), n_boot=50)
    print(f"  bootstrap     one-class slice -> {deg} (want [None, None])")
    ok &= deg == [None, None] and _f(float("nan")) is None and _f(0.123456) == 0.1235
    br = binary_rescore([[8, 1, 1, 0], [2, 5, 3, 0], [1, 4, 5, 0], [0, 0, 0, 0]])
    print(f"  detect-only   Se {br['se']} (want 0.85 - cross-type errors now count)")
    ok &= abs(br["se"] - 0.85) < 1e-9

    shutil.rmtree(tmp, ignore_errors=True)
    print("\n  SELFTEST", "PASS" if ok else "FAIL")
    return 0 if ok else 1


# ================================================================== driver
def run(dataset, root, ckpt, out_dir, level="event", icbhi_audio=None, icbhi_split=None,
        limit=None, batch_size=64, n_boot=2000, probe=False, verbose=True, icbhi_ref=None):
    os.makedirs(out_dir, exist_ok=True)
    model, cfg, meta = load_checkpoint(ckpt)
    print(f"  checkpoint {meta['path']} (row {meta['row']}, epoch {meta['epoch']}, "
          f"reported ICBHI {meta['reported_icbhi_score']:.4f})")
    print(f"  preprocessing: n_mels={cfg['n_mels']} dur={cfg['duration_s']}s "
          f"pad={cfg['padding']} minmax={cfg['minmax']} ampnorm={cfg.get('ampnorm')} "
          f"bandpass={cfg.get('bandpass')} denoise={cfg.get('denoise')}")

    if icbhi_ref is not None:
        print(f"  in-domain reference {icbhi_ref:.4f} supplied by the caller "
              "(gate already passed this session).")
    elif icbhi_audio and icbhi_split:
        icbhi_ref = verify_on_icbhi(model, cfg, meta, icbhi_audio, icbhi_split,
                                    batch_size=batch_size)
    else:
        icbhi_ref = round(float(meta["reported_icbhi_score"]), 4)
        print("  ICBHI verification SKIPPED - the in-domain reference is the checkpoint's "
              "own stored score, not a reproduction.")

    if dataset == "sprsound":
        rows, stats = sprsound_index(root, level=level)
        name, mid = "SPRSound", f"M49_SPRSound_{level}"
    elif dataset == "hflung":
        rows, stats = hflung_index(root)
        name, mid, level = "HF_Lung_V1", "M49_HF_Lung_V1_cycle", "respiratory_cycle (I+E)"
    else:
        raise ValueError(dataset)
    if limit:
        rows = rows[:limit]
        print(f"  --limit {limit}: SMOKE RUN, not a reportable result")

    y = np.array([r["label"] for r in rows])
    got = predict(model, rows, cfg, batch_size=batch_size, want_features=probe)
    pred = got["logits"].argmax(1)

    extra = {}
    if probe:
        print("  frozen-feature probe (grouped 5-fold CV) ...")
        extra["feature_probe"] = feature_probe(
            got["features"], y, np.array([r["patient_id"] for r in rows]))
    doc = make_results(mid, name, level, rows, y, pred, cfg, meta, stats, icbhi_ref,
                       got["seconds"], extra)
    if limit:
        doc["meta"]["notes"] = (f"SMOKE RUN on the first {limit} rows - NOT reportable. "
                                + doc["meta"]["notes"])

    out = os.path.join(out_dir, f"results_{mid}.json")
    json.dump(doc, open(out, "w"), indent=2)
    np.save(os.path.join(out_dir, f"preds_{mid}.npy"),
            {"y_true": y, "y_pred": pred, "logits": got["logits"],
             "group": np.array([r["patient_id"] for r in rows]),
             "stem": np.array([r["stem"] for r in rows])}, allow_pickle=True)
    plot_confusion(doc["best_metrics"]["confusion_matrix_raw"],
                   f"{name} ({level}) - zero-shot from ICBHI",
                   os.path.join(out_dir, f"confusion_{mid}.png"))
    if verbose:
        print_summary(doc)
        print(f"  wrote {out}")
    return doc


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--dataset", choices=["sprsound", "hflung"])
    ap.add_argument("--root")
    ap.add_argument("--ckpt", default=os.path.abspath(
        os.path.join(HERE, "..", "Asif's", "M45", "best_M45_P3.pth")))
    ap.add_argument("--level", default="event", choices=["event", "record"])
    ap.add_argument("--out_dir", default=HERE)
    ap.add_argument("--icbhi_audio")
    ap.add_argument("--icbhi_split", default=os.path.abspath(
        os.path.join(HERE, "..", "Asif's", "ICBHI_challenge_train_test.txt")))
    ap.add_argument("--limit", type=int)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--n_boot", type=int, default=2000)
    ap.add_argument("--probe", action="store_true")
    a = ap.parse_args()
    if a.selftest:
        return selftest()
    if not (a.dataset and a.root):
        ap.error("--dataset and --root are required (or use --selftest)")
    run(a.dataset, a.root, a.ckpt, a.out_dir, a.level, a.icbhi_audio, a.icbhi_split,
        a.limit, a.batch_size, a.n_boot, a.probe)
    return 0


if __name__ == "__main__":
    sys.exit(main())
'''

with open('/kaggle/working/m49_xval.py', 'w', encoding='utf-8') as fh:
    fh.write(M49_XVAL_SRC)
print('wrote m49_xval.py', len(M49_XVAL_SRC), 'chars')


## 4. Self-test on synthetic corpora --- runs before any real data

Builds a fake SPRSound (wav + JSON) and a fake HF_Lung_V1 (wav + `_label.txt`) in a temp
directory and pushes both through the real index builders, the real spectrogram function and
the real network. It checks the things that fail *silently*: that every label string lands in
the right ICBHI class, that an inhalation and its exhalation are paired into one cycle with
the right times, that an adventitious span outside a cycle does not label it, that the two
slices of one session share a bootstrap group, that an unknown label **raises** instead of
being bucketed into Normal, and that the metric being applied is the official one rather than
the inflated macro variant.

In [ ]:
sys.path.insert(0, "/kaggle/working")
import m49_xval as X

assert X.selftest() == 0, "self-test failed - do not run the evaluation"


## 5. The gate --- reproduce the checkpoint's own ICBHI score

If this cell does not reproduce the score stored inside the checkpoint to within 1e-3, every
number after it is uninterpretable and the notebook stops here.

In [ ]:
model, cfg, meta = X.load_checkpoint(CKPT)
print("checkpoint:", meta)
print("preprocessing:", {k: cfg[k] for k in
      ("n_mels", "duration_s", "padding", "minmax", "bandpass", "denoise", "ampnorm",
       "pretrained")})

ICBHI_REF = X.verify_on_icbhi(model, cfg, meta, ICBHI_AUDIO, ICBHI_SPLIT, batch_size=64)


## 6. Cycle level --- the headline number

One forward pass over every inhalation+exhalation cycle. Read the printed label vocabulary and
the drop counts before the score: they are the audit that the cycle builder saw what the
corpus actually contains.

In [ ]:
doc_cycle = X.run("hflung", HFL_ROOT, CKPT, WORK,
                  icbhi_ref=ICBHI_REF, limit=SMOKE, batch_size=64, probe=RUN_PROBE)


## Collect

Download `M49_results.zip` from the Output panel, unzip it into `M49_cross_dataset/`, and
commit. The results JSONs follow `Model_Training_Protocol.md` section 4, carry the full
taxonomy mapping in `dataset_info.label_mapping`, and name the bootstrap's grouping unit in
`best_metrics.icbhi_score_official_ci95_unit`.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/M49_results", "zip", WORK)
print("zipped:", os.path.getsize("/kaggle/working/M49_results.zip"), "bytes")
for f in sorted(glob.glob(os.path.join(WORK, "results_M49_*.json"))):
    d = json.load(open(f))
    b, t = d["best_metrics"], d["transfer"]
    print(f"{d['meta']['model_id']:26s} n={d['dataset_info']['test_samples']:6d}  "
          f"ICBHI {X._s(b['icbhi_score_official'])} {b['icbhi_score_official_ci95']}  "
          f"Se {X._s(b['icbhi_se_official'])}  Sp {X._s(b['icbhi_sp_official'])}  "
          f"delta vs in-domain {X._s(t['delta'], sign=True)}")
